In [3]:
import cloudcatalog
import boto3
import dask
import io
import logging
import time
import re
import pickle
import numpy as np
from astropy.io import fits
import astropy.io.fits
from dask.distributed import Client
from dask_gateway import Gateway, GatewayCluster
import math
import dask.bag as db
import s3fs
from dask.distributed import get_worker
import fsspec
from concurrent.futures import ThreadPoolExecutor, as_completed

In [4]:
fr=cloudcatalog.CloudCatalog("s3://gov-nasa-hdrl-data1/")
frID = "aia_0094"
start, stop = '2020-01-01T00:00:00Z', '2020-12-31T23:59:59Z'
file_registry1 = fr.request_cloud_catalog(frID, start_date=start, stop_date=stop, overwrite=False)
filelist = file_registry1['datakey'].to_list()

testing = False
if testing:
    s3_files = filelist[0:1000] # small test set to test
else:
    s3_files = filelist
print(len(s3_files))

# number of workers to use, for automatic scaling, our max number
n_workers = 40 # 10-50 works well, more workers actually went slower
# memory per worker (in Gb), typically 1, 2 or 4GB
w_memory = 2
# cores per worker, must be 1-4
w_cores = 4

# from the daskhub tutorial, setting up dask
gateway = Gateway()
options = gateway.cluster_options()
options.worker_cores = w_cores
options.worker_memory = w_memory

# initialize cluster and create client, takes < 15 seconds

cluster = gateway.new_cluster(options)
client = cluster.get_client() # can also use 'client=Client(cluster)'
#cluster.adapt(minimum=10, maximum=n_workers)
cluster.scale(n_workers)

# This calls the widget
cluster

IndexError: index 1 is out of bounds for axis 0 with size 1

In [ ]:
def get_irradiance(arr):
    # fast path: mean on the raw image array
    return float(arr.mean())
    
# --- single-file processing (no prints inside tasks) ---
def process_one_file(fs, s3url):
    # returns a dict
    # s3fs provides a seekable, streaming file-like object
    with fs.open(s3url, mode="rb") as f:
        with fits.open(f, memmap=False) as hdul:
            hdu = hdul[1]                 # adjust if your image HDU differs
            date = hdu.header.get("T_OBS")
            irrad = get_irradiance(hdu.data)
            return {"t_obs": date, "irradiance": irrad, "s3url": s3url}

def process_partition(urls):
    fs = s3fs.S3FileSystem(
            anon=False,
            default_fill_cache=False,
            config_kwargs={"max_pool_connections": 128},
            )

    out = []
    for u in urls:
        row = process_one_file(fs, u)
        if row is not None:
            out.append(row)
    return out

## Simplecache method

In [ ]:
def get_irradiance(arr):
    # fast path: mean on the raw image array
    return float(arr.mean())

def process_one_file(s3url):
    # cache each object to a local temp file (per worker), then read at disk speed
    url = "simplecache::" + s3url
    with fsspec.open(
        url,
        mode="rb",
        s3={
            "anon": False,
            "config_kwargs": {"max_pool_connections": 512},  # try 512; 256–1024 are common
        },
        simplecache={"cache_storage": "/tmp/dask-simplecache", "same_names": True},
    ) as f:
        with fits.open(f, memmap=False, do_not_scale_image_data=True) as hdul:
            hdu = hdul[1]  # your image HDU
            hdr = hdu.header

            # Read raw array fast
            data = np.asanyarray(hdu.data)

            # Handle missing-data sentinel if present
            blank = hdr.get("BLANK")
            if blank is not None:
                # mask BLANK values
                valid = data != blank
                if not np.any(valid):
                    return {"t_obs": None, "irradiance": None, "s3url": s3url, "error": "all-blank"}
                mean_raw = data[valid].mean(dtype=np.float64)
            else:
                mean_raw = data.mean(dtype=np.float64)

            # Get scale/offset (prefer BSCALE/BZERO; fall back to ZSCALE/ZZERO for compressed images)
            scale = hdr.get("BSCALE", hdr.get("ZSCALE", 1.0))
            zero  = hdr.get("BZERO",  hdr.get("ZZERO", 0.0))

            # Apply scaling to the mean only (cheap, keeps the speed)
            irrad = float(scale * mean_raw + zero)

            return {"t_obs": hdr.get("T_OBS"), "irradiance": irrad, "s3url": s3url}

CONCURRENCY_PER_TASK = 4 # try 4; if you see throttling, drop to 3

def process_partition(urls): 
    out = [] 
    for u in urls: 
        try: 
            row = process_one_file(u) # no per-partition S3 client needed now 
            if row is not None: 
                out.append(row) 
        except Exception as e: 
            out.append({"t_obs": None, "irradiance": None, "s3url": u, "error": str(e)}) 
    return out

In [ ]:
s3_files = filelist[0:1000] # small test set to test
print(len(s3_files))

PARTITION_SIZE = 25
print(PARTITION_SIZE)

b = db.from_sequence(s3_files, partition_size=PARTITION_SIZE)

t0 = time.time()
results = b.map_partitions(process_partition).compute()
elapsed = time.time() - t0

print(f"Processed {len(results)} records from len(s3_files) files in {elapsed:.1f}s")
# results is: [{"t_obs": "...", "irradiance": 123.45, "s3url": "s3://..."}, ...]

In [ ]:
irradiances = [r["irradiance"] for r in results if r.get("irradiance") is not None]
print("irradiance count:", len(irradiances))
irradiances[::100]

#errors = [r["error"] for r in results if r.get("error") is not None]
#errors[::100]

In [ ]:
import s3fs, numpy as np
fs = s3fs.S3FileSystem(anon=False)

# sum ContentLength cheaply
sizes = []
for u in filelist[:1000]:
    # u like 's3://bucket/key...'
    _, _, path = u.partition("s3://")
    info = fs.info(path)  # {'Size': ...} for objects
    sizes.append(info.get("size", 0))

total_bytes = int(np.sum(sizes))
mb = total_bytes / (1024**2)
print(f"Total size: {mb:.1f} MB")

In [ ]:
u = filelist[:1000][0]

_, _, path = u.partition("s3://")
#path
info = fs.info(path)  # {'Size': ...} for objects
#info
info.get("size", 0)

In [ ]:
# --- new imports ---
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import dask

# modest CPU savings (tiny dicts -> no need to compress)
dask.config.set({"distributed.comm.compression": None})

def process_one_file(s3url):
    url = "simplecache::" + s3url
    with fsspec.open(
        url,
        mode="rb",
        s3={"anon": False},
        simplecache={"cache_storage": "/tmp/dask-simplecache", "same_names": True},
    ) as f:
        # Don't let FITS rescale; we'll just take the mean of the raw array
        with fits.open(f, memmap=False, do_not_scale_image_data=True) as hdul:
            hdu = hdul[1]  # your files keep the image in HDU 1
            data = np.asanyarray(hdu.data)
            date = hdu.header.get("T_OBS")
            irrad = float(np.nanmean(data))
            return {"t_obs": date, "irradiance": irrad, "s3url": s3url}

# overlap a few downloads per task (safe, bounded)
CONCURRENCY_PER_TASK = 4

def process_partition(urls):
    out = []
    with ThreadPoolExecutor(max_workers=CONCURRENCY_PER_TASK) as ex:
        futs = {ex.submit(process_one_file, u): u for u in urls}
        for fut in as_completed(futs):
            u = futs[fut]
            try:
                row = fut.result()
            except Exception as e:
                row = {"t_obs": None, "irradiance": None, "s3url": u, "error": str(e)}
            # ensure we only append dicts
            out.append(row if isinstance(row, dict) else {
                "t_obs": None, "irradiance": None, "s3url": u,
                "error": f"non-dict-{type(row).__name__}"})
    return out

# keep your 1k test
s3_files = filelist[:1000]
PARTITION_SIZE = 25   # ~40 tasks; good with 20 workers and some intra-task threads

b = db.from_sequence(s3_files, partition_size=PARTITION_SIZE)

t0 = time.time()
# FLATTEN because process_partition returns a list of dicts
results = b.map_partitions(process_partition).compute()
elapsed = time.time() - t0

print(f"Processed {len(results)} records from {len(s3_files)} files in {elapsed:.1f}s")

# quick sanity
irradiances = [r["irradiance"] for r in results if r.get("irradiance") is not None]
print("irradiance count:", len(irradiances))

In [ ]:
import fsspec
import numpy as np
from astropy.io import fits

def process_one_file(s3url):
    url = "simplecache::" + s3url
    try:
        with fsspec.open(
            url,
            mode="rb",
            s3={"anon": False},
            simplecache={"cache_storage": "/tmp/dask-simplecache", "same_names": True},
        ) as f:
            with fits.open(f, memmap=False, do_not_scale_image_data=True) as hdul:
                # explicitly use HDU 1, which is COMPRESSED_IMAGE
                hdu = hdul[1]
                data = np.asanyarray(hdu.data)
                if data is None or data.size == 0:
                    return {"t_obs": None, "irradiance": None, "s3url": s3url, "error": "no-data"}
                date = hdu.header.get("T_OBS")
                irrad = float(np.nanmean(data))
                return {"t_obs": date, "irradiance": irrad, "s3url": s3url}
    except Exception as e:
        return {"t_obs": None, "irradiance": None, "s3url": s3url, "error": str(e)}

def process_partition(urls):
    with ThreadPoolExecutor(max_workers=CONCURRENCY_PER_TASK) as ex:
        futs = {ex.submit(process_one_file, u): u for u in urls}
        for fut in as_completed(futs):
            u = futs[fut]
            try:
                row = fut.result()
            except Exception as e:
                row = {"t_obs": None, "irradiance": None, "s3url": u, "error": str(e)}
            yield row if isinstance(row, dict) else {"t_obs": None, "irradiance": None, "s3url": u,
                                                     "error": f"non-dict-{type(row).__name__}", "raw": repr(row)}


results = b.map_partitions(process_partition).flatten().compute()
irradiances = [r["irradiance"] for r in results if isinstance(r, dict) and r.get("irradiance") is not None]
print(f"{len(irradiances)} irradiances extracted")

In [ ]:
print(results[:2])

In [ ]:
import fsspec
from astropy.io import fits

u0 = filelist[0]  # or any known-good file

with fsspec.open(
    "simplecache::" + u0, mode="rb",
    s3={"anon": False},
    simplecache={"cache_storage": "/tmp/dask-simplecache", "same_names": True},
) as f:
    with fits.open(f, memmap=False) as hdul:
        hdul.info()  # <- look at the extnames & which have ImageHDU/CompImageHDU

In [ ]:
irradiances = [r["irradiance"] for r in results if r.get("irradiance") is not None]

irradiances[::100]

In [ ]:
errors = [r["error"] for r in results if r.get("error") is not None]

errors[::100]

In [ ]:
def _flatten_to_dicts(obj):
    out = []
    stack = [obj]
    while stack:
        v = stack.pop()
        if isinstance(v, dict):
            out.append(v)
        elif isinstance(v, (list, tuple)):
            stack.extend(v)  # flatten nested partitions
        else:
            # wrap any odd value as an error dict so downstream code won't crash
            out.append({"irradiance": None, "t_obs": None, "s3url": None,
                        "error": f"unexpected-{type(v).__name__}", "raw": repr(v)})
    return out

# If you currently do:
# results = b.map_partitions(process_partition).flatten().compute()
# ...you can also do this (works either way):
raw_results = b.map_partitions(process_partition).compute()

results = _flatten_to_dicts(raw_results)

# now this is SAFE:
irradiances = [r["irradiance"] for r in results
               if isinstance(r, dict) and r.get("irradiance") is not None]
irradiances[::100]
#print("irradiances count:", len(irradiances))

In [ ]:
cluster.shutdown()

In [ ]:
[1.14938653,
 1.01091245,
 1.13348732,
 1.17269864,
 1.10908821,
 1.09712522,
 1.222366,
 1.12331226,
 1.05843719,
 1.17309558]